In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from theme import set_theme
from tests import (
    select_contracts,
    adf_by_contract,
    fisher_combine,
    engle_granger_by_contract,
    coint_by_contract,
)
set_theme()

{'rc_params': {'figure.facecolor': '#282c34',
  'axes.facecolor': '#282c34',
  'savefig.facecolor': '#282c34',
  'axes.edgecolor': '#3e4451',
  'axes.labelcolor': '#abb2bf',
  'text.color': '#abb2bf',
  'xtick.color': '#abb2bf',
  'ytick.color': '#abb2bf',
  'axes.grid': True,
  'grid.color': '#3e4451',
  'grid.linewidth': 0.6,
  'grid.linestyle': '-',
  'font.size': 9},
 'palette': ['#61afef', '#98c379', '#e5c07b', '#d19a66', '#c678dd', '#e06c75']}

## Exploration of YM Arb
##### panel dataset = all combined unadjusted series across all trading months
##### liquid dataset = continuous series of most liquid front month, rolled on the 20th of the month prior to expiry, according to open interest

In [3]:
# import ym data set
panel = pd.read_parquet(r"C:\Users\Amin\PycharmProjects\Asia Ags\data\processed\ym_corn_arb_panel.parquet")
liquid = pd.read_parquet(r"C:\Users\Amin\PycharmProjects\Asia Ags\data\processed\ym_corn_arb_liquid.parquet")

In [13]:
# test for stationarity in the stacked series, eliminating the roll and testing per expiry month

sample = select_contracts(panel)
results = adf_by_contract(sample, "ym_corn_arb_log")

combined = fisher_combine(results.p_value)
n_revert = int((results.p_value < 0.05).sum())

if combined["p_value"] < 0.05:
    verdict = "The gap does reliably close. There is a level to trade against."
else:
    verdict = ("The gap does NOT reliably close. It wanders, so there is no\n"
               "         dependable level for it to snap back to.")

print("DOES THE SA-vs-CHICAGO PRICE GAP KEEP RETURNING TO A NORMAL LEVEL?")
print("=" * 66)
print("Why this matters: you can only trade a gap that stretches wide and")
print("then closes again. If the gap simply drifts wherever it likes, there")
print("is no 'normal' level to bet on it returning to, and no trade.")
print()
print(f"  Delivery contracts examined      {len(results)}")
print(f"  Typical history per contract     {int(results.n.median())} trading days")
print(f"  Contracts where the gap closes   {n_revert} of {len(results)}")
print(f"  All contracts combined, p-value  {combined['p_value']:.3f}   (below 0.05 = it closes)")
print()
print("VERDICT:", verdict)

DOES THE SA-vs-CHICAGO PRICE GAP KEEP RETURNING TO A NORMAL LEVEL?
Why this matters: you can only trade a gap that stretches wide and
then closes again. If the gap simply drifts wherever it likes, there
is no 'normal' level to bet on it returning to, and no trade.

  Delivery contracts examined      29
  Typical history per contract     168 trading days
  Contracts where the gap closes   0 of 29
  All contracts combined, p-value  0.995   (below 0.05 = it closes)

VERDICT: The gap does NOT reliably close. It wanders, so there is no
         dependable level for it to snap back to.


In [5]:
sample["d_log_safex"] = sample.groupby("expiry")["log_safex"].diff()
sample["d_log_cbot"] = sample.groupby("expiry")["log_cbot"].diff()

In [14]:
print("SANITY CHECK 1 OF 2: DO THE TWO PRICES WANDER ON THEIR OWN?")
print("=" * 66)
print("Why we check: before asking whether the two markets travel together,")
print("we confirm each price by itself has no fixed level it returns to.")
print("Wandering is the normal, expected behaviour for a commodity price.")
print()
for column, label in [("log_safex", "SA maize (SAFEX)"), ("log_cbot", "Chicago corn")]:
    results = adf_by_contract(sample, column)
    combined = fisher_combine(results.p_value)
    n_anchored = int((results.p_value < 0.05).sum())
    print(f"  {label:<20} anchored to a level in {n_anchored} of {len(results)} contracts"
          f"  (p-value {combined['p_value']:.2f})")
print()
print("Reading it: a p-value ABOVE 0.05 means 'no fixed level'. Both prices")
print("wander freely, exactly as expected. The setup is sound.")

SANITY CHECK 1 OF 2: DO THE TWO PRICES WANDER ON THEIR OWN?
Why we check: before asking whether the two markets travel together,
we confirm each price by itself has no fixed level it returns to.
Wandering is the normal, expected behaviour for a commodity price.

  SA maize (SAFEX)     anchored to a level in 1 of 29 contracts  (p-value 0.70)
  Chicago corn         anchored to a level in 2 of 29 contracts  (p-value 0.64)

Reading it: a p-value ABOVE 0.05 means 'no fixed level'. Both prices
wander freely, exactly as expected. The setup is sound.


In [15]:
print("SANITY CHECK 2 OF 2: ARE THE DAY-TO-DAY MOVES WELL BEHAVED?")
print("=" * 66)
print("Why we check: the prices wander, but their daily CHANGES should be")
print("stable - roughly the same size day after day, with no runaway drift.")
print("This confirms the data is clean enough for the tests that follow.")
print()
for column, label in [("d_log_safex", "SA maize (SAFEX)"), ("d_log_cbot", "Chicago corn")]:
    results = adf_by_contract(sample, column)
    combined = fisher_combine(results.p_value)
    n_stable = int((results.p_value < 0.05).sum())
    shown_p = f"{combined['p_value']:.2f}" if combined["p_value"] >= 0.01 else "under 0.01"
    print(f"  {label:<20} daily moves stable in {n_stable} of {len(results)} contracts"
          f"  (p-value {shown_p})")
print()
print("Reading it: here a p-value BELOW 0.05 is what we want, and every")
print("contract passes. Prices wander, daily moves are stable - textbook.")

SANITY CHECK 2 OF 2: ARE THE DAY-TO-DAY MOVES WELL BEHAVED?
Why we check: the prices wander, but their daily CHANGES should be
stable - roughly the same size day after day, with no runaway drift.
This confirms the data is clean enough for the tests that follow.



KeyError: 'd_log_safex'

In [16]:
betas, _ = engle_granger_by_contract(sample)
coint_results = coint_by_contract(sample)

MONTH_NAMES = {3: "March", 7: "July", 12: "December"}
combined = fisher_combine(coint_results.p_value)
n_linked = int((coint_results.p_value < 0.05).sum())

print("HOW TIGHTLY IS SA MAIZE TIED TO CHICAGO CORN?")
print("=" * 66)
print("Two separate questions. First, when Chicago moves, how much does SA")
print("move with it? Second, are the two tethered together over the long")
print("run, or can they drift apart indefinitely?")
print()
print("1. WHEN CHICAGO CORN MOVES 10%, SA MAIZE MOVES:")
print(f"     Typical contract        {betas['beta'].median() * 10:.1f}%")
print(f"     Weakest contract        {betas['beta'].min() * 10:.1f}%")
print(f"     Strongest contract      {betas['beta'].max() * 10:.1f}%")
print("   Some contracts move the OPPOSITE way to Chicago (negative above),")
print("   which means local SA supply is driving the price, not the world.")
print()
print("2. HOW MUCH OF SA'S PRICE MOVEMENT DOES CHICAGO EXPLAIN?")
print(f"     Typical contract        {betas['r_squared'].median():.0%}")
print(f"     Best contract           {betas['r_squared'].max():.0%}")
print(f"     Worst contract          {betas['r_squared'].min():.0%}")
print("   So in a typical year most of what moves SA maize is local.")
print()
print("   By delivery month (typical contract):")
for month, row in betas.groupby("delivery_month")[["beta", "r_squared"]].median().iterrows():
    print(f"     {MONTH_NAMES.get(month, month):<12} SA moves {row['beta'] * 10:.1f}% per 10% Chicago move,"
          f"  Chicago explains {row['r_squared']:.0%}")
print()
print("3. ARE THE TWO MARKETS TETHERED OVER THE LONG RUN?")
print(f"     Contracts that are tethered   {n_linked} of {len(coint_results)}")
print(f"     All contracts combined        p-value {combined['p_value']:.3f}   (below 0.05 = tethered)")
print()
print("VERDICT: SA maize and Chicago corn are NOT tethered. They can drift")
print("         apart and stay apart. Chicago is a loose influence on SA")
print("         prices, not an anchor that pulls them back.")

HOW TIGHTLY IS SA MAIZE TIED TO CHICAGO CORN?
Two separate questions. First, when Chicago moves, how much does SA
move with it? Second, are the two tethered together over the long
run, or can they drift apart indefinitely?

1. WHEN CHICAGO CORN MOVES 10%, SA MAIZE MOVES:
     Typical contract        4.7%
     Weakest contract        -5.4%
     Strongest contract      18.6%
   Some contracts move the OPPOSITE way to Chicago (negative above),
   which means local SA supply is driving the price, not the world.

2. HOW MUCH OF SA'S PRICE MOVEMENT DOES CHICAGO EXPLAIN?
     Typical contract        29%
     Best contract           94%
     Worst contract          1%
   So in a typical year most of what moves SA maize is local.

   By delivery month (typical contract):
     March        SA moves 5.2% per 10% Chicago move,  Chicago explains 30%
     July         SA moves 4.6% per 10% Chicago move,  Chicago explains 22%
     December     SA moves 4.7% per 10% Chicago move,  Chicago explains 49%

In [17]:
betas["year"] = pd.to_datetime(betas["expiry"]).dt.year
ranked = betas.sort_values("r_squared", ascending=False)

readable = pd.DataFrame({
    "Contract": ranked["expiry"],
    "Trading days": ranked["n"],
    "SA move per 10% Chicago move": (ranked["beta"] * 10).round(1).astype(str) + "%",
    "Chicago explains": (ranked["r_squared"] * 100).round(0).astype(int).astype(str) + "%",
})

print("CONTRACT BY CONTRACT, STRONGEST LINK TO CHICAGO FIRST")
print("=" * 66)
print("Each row is one delivery contract. The last column is the share of")
print("SA price movement that Chicago explains - high means SA was tracking")
print("the world market, low means SA was trading on its own local story.")
print()
print(readable.to_string(index=False))
print()
print("Notice the pattern: 2020-2022 sit at the top, when global grain")
print("markets moved together during covid and the Ukraine invasion. The")
print("bottom of the table is dominated by SA drought and surplus years,")
print("where the local crop set the price and Chicago was irrelevant.")

CONTRACT BY CONTRACT, STRONGEST LINK TO CHICAGO FIRST
Each row is one delivery contract. The last column is the share of
SA price movement that Chicago explains - high means SA was tracking
the world market, low means SA was trading on its own local story.

Contract  Trading days SA move per 10% Chicago move Chicago explains
 2022-07           209                         8.4%              94%
 2021-07           293                         6.9%              94%
 2020-07           236                         8.7%              90%
 2020-12           163                        15.4%              85%
 2022-03           161                         4.9%              82%
 2023-07           202                        18.6%              80%
 2021-03           220                         8.3%              75%
 2020-03           179                         7.9%              69%
 2016-12           153                         3.2%              54%
 2025-03           164                        11.8% 

In [18]:
cbot_vol = sample.groupby("expiry")["d_log_cbot"].std()
betas = betas.merge(cbot_vol.rename("cbot_vol"), on="expiry")

link_vs_vol = betas[["r_squared", "cbot_vol"]].corr().iloc[0, 1]
link_vs_beta = betas[["r_squared", "beta"]].corr().iloc[0, 1]

print("WHEN DOES SA ACTUALLY FOLLOW CHICAGO?")
print("=" * 66)
print("The link is not constant - it comes and goes. These two numbers say")
print("what it travels with. They run from -1 to +1; further from zero means")
print("a stronger pattern, and positive means the two rise together.")
print()
print(f"  Link strength vs how choppy Chicago was    {link_vs_vol:+.2f}")
print(f"  Link strength vs how hard SA follows       {link_vs_beta:+.2f}")
print()
print("What this says: SA maize tracks Chicago most closely in the periods")
print("when Chicago itself is moving violently. In calm global markets the")
print("connection fades and SA trades on local supply alone.")
print()
print("The practical problem: the link is strongest exactly when world grain")
print("markets are in crisis, which is when a spread trade is most dangerous")
print("to hold, and it is absent in the quiet periods you would want to trade.")

KeyError: 'Column not found: d_log_cbot'

In [19]:
arb_range = sample.groupby("expiry")["ym_corn_arb_log"].agg(lambda s: s.max() - s.min())
betas = betas.merge(arb_range.rename("arb_range"), on="expiry")

link_vs_range = betas[["r_squared", "arb_range"]].corr().iloc[0, 1]

print("THE CATCH: THE BIGGEST OPPORTUNITIES ARE THE LEAST RELIABLE")
print("=" * 66)
print("A spread trade needs the gap to swing enough to be worth trading.")
print("So we check whether the contracts with the widest swings are also")
print("the ones where the two markets are most closely linked.")
print()
print(f"  Link strength vs size of the swing    {link_vs_range:+.2f}")
print()
print("The sign is negative, and that is the problem in one number. The")
print("contracts where the gap swings widest - the ones that look most")
print("profitable - are exactly the contracts where SA and Chicago are")
print("LEAST connected. Those wide swings are SA local supply shocks, not")
print("a gap that is going to close.")
print()
print("BOTTOM LINE ACROSS ALL THE TESTS ABOVE:")
print("  - The SA-vs-Chicago gap does not reliably close.")
print("  - The two markets are not tethered over the long run.")
print("  - Where the gap moves most, the connection is weakest.")
print("  On this evidence, trading this spread directly is not supported.")

THE CATCH: THE BIGGEST OPPORTUNITIES ARE THE LEAST RELIABLE
A spread trade needs the gap to swing enough to be worth trading.
So we check whether the contracts with the widest swings are also
the ones where the two markets are most closely linked.

  Link strength vs size of the swing    -0.30

The sign is negative, and that is the problem in one number. The
contracts where the gap swings widest - the ones that look most
profitable - are exactly the contracts where SA and Chicago are
LEAST connected. Those wide swings are SA local supply shocks, not
a gap that is going to close.

BOTTOM LINE ACROSS ALL THE TESTS ABOVE:
  - The SA-vs-Chicago gap does not reliably close.
  - The two markets are not tethered over the long run.
  - Where the gap moves most, the connection is weakest.
  On this evidence, trading this spread directly is not supported.
